# 01 – Data cleaning (PoQeMoN QoE project)

**Goal of this notebook:** load `pokemon.csv`, fix the problems we found in the data exploration, and save a clean file `pokemon_clean.csv` for the modelling notebooks.

**Cleaning rules (decided after exploration):**

| # | Problem | Decision |
|---|---|---|
| 1 | 1 duplicate row | Remove it |
| 2 | Resolution = 16 (impossible value) | Remove that row |
| 3 | Buffering time is very skewed (max ≈ 329,000) | Use log of buffering time |
| 4 | 6 phone models with only 1 session | Group them as `other` |
| 5 | `id` is only a row number, `QoD_os-version` repeats model + api-level | Remove these columns |
| 6 | QoF columns = user's own ratings → data leakage | Keep them in the file, but **not** in the main feature list |

Category columns (network type, operator, phone model) are **not** one-hot encoded here. We do it later inside the model pipeline (see the end of this notebook).

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42          # the same seed for the whole project
np.random.seed(RANDOM_SEED)

## 0. Load the data
- **On your computer (VS Code / Jupyter):** put `pokemon.csv` in the same folder as this notebook.
- **In Google Colab:** if `pokemon.csv` is not in the files panel yet, the cell asks you to upload it.

In [ ]:
DATA_FILE = "pokemon.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files   # only exists in Google Colab
        files.upload()                   # choose pokemon.csv on your computer
    except ImportError:
        raise FileNotFoundError("Put pokemon.csv in the same folder as this notebook (see README).")

df = pd.read_csv(DATA_FILE)
print("Rows, columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
df.head()

## Rule 1 – Remove the duplicate row
Two rows have exactly the same values in every column except `id`. The same session counted twice would give it double weight, so we keep only one.

In [ ]:
cols_without_id = [c for c in df.columns if c != "id"]
print("Duplicate rows:", df.duplicated(subset=cols_without_id).sum())

df = df.drop_duplicates(subset=cols_without_id).reset_index(drop=True)
print("Rows now:", len(df))

## Rule 2 – Remove the impossible resolution
Resolution should be 240 or 360 (pixels). One row has 16, which is a measurement error.

In [ ]:
print(df["QoA_VLCresolution"].value_counts())

df = df[df["QoA_VLCresolution"].isin([240, 360])].reset_index(drop=True)
print("Rows now:", len(df))

## Rule 3 – Log of buffering time
Most buffering times are about 2,000–4,000 (probably milliseconds; the unit is not given in the documentation), but a few are huge. These extreme values would have too much influence, especially on linear regression.
The **log** makes big values smaller and keeps the order (bigger stays bigger). The two histograms show the difference.

In [ ]:
df["log_BUFFERINGtime"] = np.log(df["QoA_BUFFERINGtime"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(df["QoA_BUFFERINGtime"], bins=50)
axes[0].set_title("Buffering time (original)")
axes[1].hist(df["log_BUFFERINGtime"], bins=50)
axes[1].set_title("log(Buffering time)")
plt.tight_layout()
plt.show()

df = df.drop(columns=["QoA_BUFFERINGtime"])

## Rule 4 – Group rare phone models
Six phone models have only 1 session each. A model cannot learn anything from one example, so we group them as `other`.

In [ ]:
counts = df["QoD_model"].value_counts()
rare_models = counts[counts < 5].index
print("Rare models:", list(rare_models))

df["QoD_model"] = df["QoD_model"].where(~df["QoD_model"].isin(rare_models), "other")
print(df["QoD_model"].value_counts())

## Rule 5 – Remove useless columns
- `id`: only a row number, no meaning.
- `QoD_os-version`: very detailed text; the same information is already in `QoD_model` and `QoD_api-level`.

We **keep** `user_id`, but only to split the data by user later. It is not used as an input.

In [ ]:
df = df.drop(columns=["id", "QoD_os-version"])
print(df.columns.tolist())

## Rule 6 – Define the column groups
This list decides what the models are allowed to see. The QoF columns are kept in a separate list, only for the extra experiment that shows the effect of data leakage.

In [ ]:
TARGET = "MOS"
GROUP = "user_id"            # used only for splitting by user

NUMERIC_FEATURES = [
    "QoA_VLCresolution", "QoA_VLCbitrate", "QoA_VLCframerate", "QoA_VLCdropped",
    "QoA_VLCaudiorate", "QoA_VLCaudioloss", "QoA_BUFFERINGcount", "log_BUFFERINGtime",
    "QoD_api-level", "QoU_sex", "QoU_age", "QoU_Ustedy",
]
CATEGORICAL_FEATURES = ["QoS_type", "QoS_operator", "QoD_model"]   # one-hot encoded later
LEAKAGE_FEATURES = ["QoF_begin", "QoF_shift", "QoF_audio", "QoF_video"]  # NOT in the main experiment

# Safety check: every column belongs to exactly one group
all_listed = NUMERIC_FEATURES + CATEGORICAL_FEATURES + LEAKAGE_FEATURES + [TARGET, GROUP]
assert sorted(all_listed) == sorted(df.columns), "A column is missing or listed twice"
print("Main features:", len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))

## Save the clean data

In [ ]:
df.to_csv("pokemon_clean.csv", index=False)

print("Saved pokemon_clean.csv")
print("Rows:", len(df), "| Users:", df[GROUP].nunique())
print("MOS distribution:")
print(df[TARGET].value_counts().sort_index())

## Summary and next step

- Rows: 1543 → **1541** (1 duplicate, 1 impossible resolution removed)
- Buffering time replaced by its log
- 6 rare phone models grouped as `other`
- `id` and `QoD_os-version` removed
- 15 main features; the 4 QoF columns are kept apart (data leakage)

**Why one-hot encoding is not done here:** we will put it inside a scikit-learn *pipeline* in the modelling notebook. Then every preparation step is applied in the same way inside each cross-validation fold, and the notebook stays simple.

**Next notebook:** `02_models.ipynb` – split the data by user, set up the baseline, train the models, and run the grid search.